In [1]:
from utils import *

import torch as th
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.nn.functional as F

import importlib
import data_handler

importlib.reload(data_handler)

import tqdm


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class RNet(nn.Module):
    def __init__(self, num_classes=2):
        super(RNet, self).__init__()
        self.resnet = models.resnet18(pretrained=False)
        
        in_features = self.resnet.fc.in_features
        #self.resnet.fc = nn.Linear(in_features, num_classes)
        
        
        self.resnet.fc = nn.Sequential(
            nn.Linear(in_features, 4096), 
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes)
            )

    def forward(self, x):
        return self.resnet(x)

# Example usage:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RNet(num_classes=2).to(device)


In [ ]:
device = th.device("cuda" if th.cuda.is_available() else "cpu")

model = RNet().to(device)


def train():
    """
    """
    # params
    epochs = 10
    lr = 0.0001
    # Loss function
    #loss_func = nn.BCEWithLogitsLoss()
    loss_func = nn.CrossEntropyLoss()

    # Optimizer and model inits
    optimizer = th.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in tqdm.tqdm(range(epochs)):
        for images, labels in data_handler.tr_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            # Forward pass
            outputs = model(images)

            # Calculate the loss
            loss = loss_func(outputs, labels)

            # Backward pass and gradient update
            loss.backward()
            optimizer.step()
            
            torch.cuda.empty_cache()

            
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")
    
    th.save(model.state_dict(), "resnet18_not_pretrained_with_dropout.pt")

In [ ]:
train()

In [ ]:
# Load the model
model = RNet().to(device)
#model.load_state_dict(th.load("resnet18_not_pretrained.pt"))
model.load_state_dict(th.load("resnet18_not_pretrained_with_dropout.pt"))

# Test the model
model.eval()

# Test loop
total_images = 0
nr_acc = 0
for images, label in data_handler.te_loader:
    images = images.to(device)
    labels = label.to(device)
    
    # Forward pass
    outputs = model(images)
    
    # Predicted classes
    predicted = th.argmax(outputs, dim=1)
    
    # Accuracy calculation (vectorized)
    nr_acc += (predicted == labels).sum().item()  # Count correct predictions
    total_images += labels.size(0)  # Keep track of the total number of images
    
    
    # Original image
    #plt.imshow(images[0].cpu().permute(1, 2, 0).numpy())
    #plt.title(f"Pred: {"1" if predicted[0] else "0"} | Label: {labels[0]}")
    #plt.axis('off')
    
    #plt.show()

print(nr_acc/total_images)